<a href="https://colab.research.google.com/github/tristanengst/FintechAndAICS983CourseStaff/blob/main/ColabNotebooks/ColabsFinGPT/1Reproduce_Benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CASE1 Reproduce Benchmark

**Purpose:**  
This part is used to reproduce the result of performance of the **FinGPT v3.2** sentiment classification models on four different financial sentiment analysis datasets.

**Benchmark Results (expected results):**  

||[Financial_Phrasebank(FPB)](https://huggingface.co/datasets/takala/financial_phrasebank)|[FiQA SA](https://huggingface.co/datasets/pauri32/fiqa-2018)|[Twitter Financial News Sentiment (TFNS)](https://huggingface.co/datasets/zeroshot/twitter-financial-news-sentiment)|[News With GPT Instruction (NWGI)](https://huggingface.co/datasets/oliverwang15/news_with_gpt_instructions)|  
|-----|-----|-----|-----|-----|
|Weighted F1|0.850|0.860|0.894|0.632|  
|ACC/F1 Micro|0.851|0.844|0.894|0.636|  
|Macro F1|0.840|0.752|0.866|0.644|




---
## Instructions

**Step 1 – Switch to A100 runtime (Colab only)**  
> Purpose: loading FinGPT v3.2 (LLaMA2-based) requires a powerful GPU italicized text, such as A100 or RTX 3090.  

1. Go to **Runtime → Change runtime type** in the top menu.  
2. Set **Hardware accelerator** to `A100 GPU` or `L4 GPU`.  
3. To access these better GPUs in Google Colab, you may need to:  
   - Upgrade to *Colab Pro*, or  
   - Use *Colab Pay-As-You-Go* (**recommended for flexibility**)  
   - [Click here](https://colab.research.google.com/signup) to upgrade or enable Pay-As-You-Go.  
4. Click **Save**.

**Step 2 – Set your API key**  
 please set your API keys as environment variables to avoid exposing sensitive information in the code.

  On the left sidebar, click the “🔑 Secrets” icon.

  Add the following secret:

1. HF_TOKEN
 - HF_TOKEN refers to your Hugging Face access token.
 - Visit: https://huggingface.co/settings/tokens

Enable "Notebook Access" on the left side of each secret.

**Step 3 – Run the entire notebook**  
Once the dependencies are installed and runtime is set, go to **Runtime → Run all** to execute all cells.

---
Works for me OS August 13


## Code

### Install dependencies

In [ ]:
!pip install -U bitsandbytes==0.46.1 datasets==3.6.0
!pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu126
!pip install huggingface_hub==0.35.3 transformers==4.57.1 peft==0.17.1 accelerate==1.10.1 tokenizers==0.22.1

Looking in indexes: https://download.pytorch.org/whl/cu126


### Clone FinNLP Repository

In [ ]:
!git clone https://github.com/AI4Finance-Foundation/FinNLP

fatal: destination path 'FinNLP' already exists and is not an empty directory.


In [ ]:
import sys
sys.path.append('/content/FinNLP')  # https://github.com/AI4Finance-Foundation/FinNLP

In [ ]:
sys.path.append('/content/FinNLP')
print(sys.path)
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM, LlamaForCausalLM, LlamaTokenizerFast   # 4.30.2
from peft import PeftModel  # 0.4.0
import torch


from finnlp.benchmarks.fpb import test_fpb
from finnlp.benchmarks.fiqa import test_fiqa , add_instructions
from finnlp.benchmarks.tfns import test_tfns
from finnlp.benchmarks.nwgi import test_nwgi

['/content', '/env/python', '/usr/lib/python313.zip', '/usr/lib/python3.13', '/usr/lib/python3.13/lib-dynload', '', '/usr/local/lib/python3.13/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.13/dist-packages/IPython/extensions', '/root/.ipython', '/content/FinNLP', '/content/FinNLP']


/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


### Load Model

In [ ]:
# v3.2
# base_model = "meta-llama/Llama-2-7b-chat-hf"  # Access needed
base_model = "daryl149/llama-2-7b-chat-hf"
peft_model = "oliverwang15/FinGPT_v32_Llama2_Sentiment_Instruction_LoRA_FT"
tokenizer = LlamaTokenizerFast.from_pretrained(base_model, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
model = LlamaForCausalLM.from_pretrained(base_model, trust_remote_code=True, device_map = "cuda:0", dtype=torch.float16)
model = PeftModel.from_pretrained(model, peft_model)
model = torch.compile(model)  # Please comment this line if your platform does not support torch.compile
model = model.eval()

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

### Test

The following cells evaluate the model’s performance on the Financial Phrasebank, FiQA SA, Twitter Financial News Sentiment (TFNS), and News With GPT Instruction (NWGI) datasets.

- **Data loading:** For each dataset, we use only the designated test portion (or a split from the available data) for evaluation.
- **Metrics:** For each dataset, the code outputs **Weighted F1**, **ACC/F1 Micro**, and **Macro F1** scores.  
  Users can directly compare these outputs with the **expected results** table above to assess performance consistency.

In [ ]:
# the model processes 8 input samples at a time
batch_size = 8

In [ ]:
# FPB
# Lates FPB repo: https://huggingface.co/datasets/takala/financial_phrasebank
# When prompted with "Do you wish to run the custom code? [y/N]", type y and press Enter to continue loading the dataset.

import importlib
import finnlp.benchmarks.fpb as fpb_mod

importlib.reload(fpb_mod)

test_fpb = fpb_mod.test_fpb

res = test_fpb(model, tokenizer, batch_size = batch_size)

Generating train split:   0%|          | 0/4846 [00:00<?, ? examples/s]



Prompt example:
Instruction: What is the sentiment of this news? Please choose an answer from {negative/neutral/positive}.
Input: L&T has also made a commitment to redeem the remaining shares by the end of 2011 .
Answer: 


Total len: 1212. Batchsize: 8. Total steps: 152


100%|██████████| 152/152 [00:55<00:00,  2.73it/s]

Acc: 0.8514851485148515. F1 macro: 0.8406449939437212. F1 micro: 0.8514851485148515. F1 weighted (BloombergGPT): 0.85095766698792. 


In [ ]:
# FiQA
import importlib
import finnlp.benchmarks.fiqa as fiqa_mod

importlib.reload(fiqa_mod)

test_fiqa = fiqa_mod.test_fiqa
res = test_fiqa(model, tokenizer, prompt_fun = add_instructions, batch_size = batch_size)

train.csv: 0.00B [00:00, ?B/s]

validation.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/961 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/102 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/150 [00:00<?, ? examples/s]



Prompt example:
Instruction: What is the sentiment of this tweet? Please choose an answer from {negative/neutral/positive}.
Input: This $BBBY stock options trade would have more than doubled your money https://t.co/Oa0loiRIJL via @TheStreet
Answer: 


Total len: 275. Batchsize: 8. Total steps: 35


100%|██████████| 35/35 [00:11<00:00,  3.16it/s]

Acc: 0.8436363636363636. F1 macro: 0.7524122647015905. F1 micro: 0.8436363636363636. F1 weighted (BloombergGPT): 0.860437084621201. 


In [ ]:
# TFNS
res = test_tfns(model, tokenizer, batch_size = batch_size)

README.md: 0.00B [00:00, ?B/s]

sent_train.csv: 0.00B [00:00, ?B/s]

sent_valid.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/9543 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2388 [00:00<?, ? examples/s]



Prompt example:
Instruction: What is the sentiment of this tweet? Please choose an answer from {negative/neutral/positive}.
Input: $ALLY - Ally Financial pulls outlook https://t.co/G9Zdi1boy5
Answer: 


Total len: 2388. Batchsize: 8. Total steps: 299


100%|██████████| 299/299 [01:42<00:00,  2.92it/s]

Acc: 0.8923785594639866. F1 macro: 0.8656153798038404. F1 micro: 0.8923785594639866. F1 weighted (BloombergGPT): 0.8925275537856603. 


In [ ]:
# NWGI
res = test_nwgi(model, tokenizer, batch_size = batch_size)

README.md:   0%|          | 0.00/682 [00:00<?, ?B/s]

data/train-00000-of-00001-dd971e407aecb3(…):   0%|          | 0.00/10.8M [00:00<?, ?B/s]

data/test-00000-of-00001-c551483ebf36549(…):   0%|          | 0.00/2.73M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16184 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/4047 [00:00<?, ? examples/s]



Prompt example:
Instruction: What is the sentiment of this news? Please choose an answer from {negative/neutral/positive}.
Input: In the latest trading session, Adobe Systems (ADBE) closed at $535.98, marking a +0.31% move from the previous day.
Answer: 


Total len: 4047. Batchsize: 8. Total steps: 506


100%|██████████| 506/506 [03:42<00:00,  2.27it/s]

Acc: 0.636767976278725. F1 macro: 0.6450563921094842. F1 micro: 0.636767976278725. F1 weighted (BloombergGPT): 0.6331439847092933. 


## **Declaration**

This notebook is adapted from the AI4Finance-Foundation/FinGPT repository([Github Link](https://github.com/AI4Finance-Foundation/FinGPT/blob/master/fingpt/FinGPT_Sentiment_Analysis_v3/benchmark/benchmarks.ipynb))